# Exploración y modelos (football dataset)

- Arranca Jupyter desde la **raíz del proyecto**: `kedro jupyter notebook` o `kedro jupyter lab`.
- Usa el kernel **Kedro (analisis_equipos_de_football)** o el intérprete **`.venv/bin/python`**.
- Necesitas `data/raw/database.sqlite` en local (no va en git).


In [ ]:
%load_ext kedro.ipython

In [ ]:
catalog

## Machine learning: resultado del partido

Clases: **victoria visitante (0)** / **empate (1)** / **victoria local (2)**.  
Features iniciales: cuotas **Bet365** (`B365H`, `B365D`, `B365A`). Luego comparamos varios algoritmos con las mismas particiones `train/test`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

match = catalog.load("match")

goal_diff = match["home_team_goal"] - match["away_team_goal"]
outcome = np.select(
    [goal_diff > 0, goal_diff == 0, goal_diff < 0],
    [2, 1, 0],
)

feat_cols = ["B365H", "B365D", "B365A"]
df = match[feat_cols].assign(outcome=outcome).dropna()

X = df[feat_cols]
y = df["outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Muestras: train={len(X_train):,}  test={len(X_test):,}")
print("Distribución de clases (train):", y_train.value_counts().sort_index().to_dict())

In [ ]:
models = {
    "LogisticRegression": Pipeline(
        [
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=2_000, solver="lbfgs")),
        ]
    ),
    "LinearSVC": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "clf",
                LinearSVC(dual=False, max_iter=5_000, random_state=42),
            ),
        ]
    ),
    "KNN_k15": Pipeline(
        [
            ("scale", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=15, weights="distance")),
        ]
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        max_depth=16,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=200,
        max_depth=6,
        learning_rate=0.08,
        random_state=42,
    ),
}

rows = []
fitted = {}
predictions = {}

for name, est in models.items():
    est.fit(X_train, y_train)
    pred = est.predict(X_test)
    fitted[name] = est
    predictions[name] = pred
    rows.append(
        {
            "modelo": name,
            "accuracy": accuracy_score(y_test, pred),
            "f1_macro": f1_score(y_test, pred, average="macro"),
            "f1_weighted": f1_score(y_test, pred, average="weighted"),
        }
    )

resultados = pd.DataFrame(rows).sort_values("f1_macro", ascending=False).reset_index(
    drop=True
)
resultados

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(resultados))
w = 0.25
ax.bar(x - w, resultados["accuracy"], width=w, label="accuracy")
ax.bar(x, resultados["f1_macro"], width=w, label="f1_macro")
ax.bar(x + w, resultados["f1_weighted"], width=w, label="f1_weighted")
ax.set_xticks(x)
ax.set_xticklabels(resultados["modelo"], rotation=20, ha="right")
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("Comparación de modelos (mismas features y split)")
plt.tight_layout()
plt.show()

In [ ]:
mejor = resultados.iloc[0]["modelo"]
y_pred_mejor = predictions[mejor]
print(f"Mejor según f1_macro: {mejor}\n")
print(
    classification_report(
        y_test,
        y_pred_mejor,
        target_names=["away_win", "draw", "home_win"],
    )
)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_mejor,
    display_labels=["away", "draw", "home"],
    ax=ax,
    colorbar=False,
)
ax.set_title(f"Matriz de confusión — {mejor}")
plt.tight_layout()
plt.show()

### Próximos pasos

- Más columnas de `match` (otras casas de apuestas) o joins con `team` / `team_attributes`.
- Validación **por temporada** (`season`) para no mezclar futuro y pasado.
- Afinar hiperparámetros con `GridSearchCV` o `RandomizedSearchCV` sobre el modelo que mejor salga aquí.
